In [1]:
%pip install nltk
%pip install websockets
%pip install playwright
%pip install transformers torch
%pip install spacy
%pip install asyncpg



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import asyncio
import websockets
import nltk
import spacy
import asyncpg
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from playwright.async_api import async_playwright
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline
from datetime import datetime

d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Resource setup
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Download VADER lexicon
nltk.download('vader_lexicon')


sia = SentimentIntensityAnalyzer()

nltk.download(['punkt', 'stopwords', 'punkt_tab', 'wordnet', 'omw-1.4'])
lemmatizer = WordNetLemmatizer()

nlp = spacy.load("en_core_web_sm")


sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 # Set to 0 if you have a GPU
)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [4]:
BRAVE_PATH = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"
AUTH_FILE = "auth.json"
MAX_CONCURRENT_TABS = 3
semaphore = asyncio.Semaphore(MAX_CONCURRENT_TABS)
word_counts = Counter()

# Refined ignore list
ignore_words = [
    "i", "me", "my", "myself", "we", "us", "our", "ours", "ourselves", "you", "your", "yours", 
    "yourself", "yourselves", "he", "him", "his", "she", "her", "hers", "it", "its", "they", 
    "them", "their", "someone", "anyone", "am", "is", "are", "was", "were", "be", "been", 
    "being", "have", "has", "had", "do", "does", "did", "shall", "will", "should", "would", 
    "may", "might", "must", "can", "could", "get", "got", "getting", "go", "goes", "went", 
    "gone", "take", "took", "taken", "make", "made", "making", "like", "know", "knows", 
    "knew", "want", "wants", "wanted", "look", "looks", "really", "very", "just", "actually", 
    "basically", "literally", "even", "also", "still", "maybe", "probably", "quite", "much", 
    "many", "lot", "lots", "thing", "things", "stuff", "reddit", "subreddit", "sub", "post", 
    "posts", "comment", "comments", "thread", "edit", "deleted", "removed", "amp", "https", 
    "http", "com", "www"
]

In [5]:
async def reanalyze_cache(subreddit):
    """
    Fetches all posts for a subreddit from PostgreSQL, 
    re-runs NLP analysis, and updates the records.
    """
    print(f"Starting database re-analysis for r/{subreddit}...")
    
    conn = await get_db_connection()
    try:
        # 1. Fetch all posts for this sub (No limit, we want to upgrade everything)
        rows = await conn.fetch(
            "SELECT id, title, body FROM reddit_posts WHERE subreddit = $1", 
            subreddit
        )
        total = len(rows)
        processed = 0
        
        for row in rows:
            # Combine title and body for full context
            title = row.get('title', '')
            body = row.get('body', '')
            full_text = f"{title} {body}"
            
            # 2. Re-run NLP Analysis using latest models
            sentiment = get_sentiment(full_text)
            
            # Keywords Extraction
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [
                lemmatizer.lemmatize(w) for w in words 
                if w.isalnum() and w not in sw and w not in ignore_words
            ]
            keywords = dict(Counter(clean))
            
            # NER Extraction (All 18 labels)
            entities = extract_entities(full_text)
            
            # 3. Update the record in the Database
            # Using json.dumps ensures the driver passes the correct type to JSONB columns
            await conn.execute('''
                UPDATE reddit_posts 
                SET sentiment = $1, keywords = $2, entities = $3 
                WHERE id = $4
            ''', sentiment, json.dumps(keywords), json.dumps(entities), row['id'])
            
            processed += 1
            if processed % 10 == 0:
                print(f"Progress: {processed}/{total} posts updated in DB...")
                
    finally:
        await conn.close()
        
    print(f"Re-analysis complete for r/{subreddit}. All data is now standardized.")

# Example Usage (Manually run this for any sub you already have data for):
# await reanalyze_cache("AskIndianWomen")

In [6]:
def extract_entities(text):
    doc = nlp(text)
    entities = []
    # We filter for categories most useful for Business Intelligence
    target_labels = [
        "PERSON", "NORP", "FAC", "ORG", "GPE", "LOC", "PRODUCT", "EVENT", 
        "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT", 
        "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"
    ]
    
    for ent in doc.ents:
        if ent.label_ in target_labels:
            entities.append({
                "text": ent.text,
                "label": ent.label_
            })
    return entities

In [7]:
def get_sentiment(text):
    """
    Scans entire text and understands context using a 
    Transformer model. Returns Positive, Neutral, or Negative.
    """
    # Truncate text to 512 tokens (Model limit)
    truncated_text = text[:512]
    result = sentiment_pipeline(truncated_text)[0]
    
    # Mapping the model labels to your format
    label_map = {
        "positive": "Positive",
        "neutral": "Neutral",
        "negative": "Negative"
    }
    return label_map.get(result['label'].lower(), "Neutral")

In [8]:


DB_CONFIG = {
    "user": "postgres",
    "password": "admin123",
    "database": "reddit-scrapper",
    "host": "127.0.0.1"
}

def safe_parse_timestamp(ts_str):
    """
    Converts Reddit ISO strings to Python datetime objects.
    Strips timezone info to prevent 'offset-naive' vs 'offset-aware' errors.
    """
    if not ts_str or ts_str == "" or ts_str == "None":
        return None
    try:
        # 1. Parse the ISO string (making it 'aware' initially)
        dt = datetime.fromisoformat(ts_str.replace('Z', '+00:00'))
        
        # 2. Strip the timezone info to make it 'naive' 
        # This matches PostgreSQL's TIMESTAMP WITHOUT TIME ZONE
        return dt.replace(tzinfo=None)
    except Exception as e:
        print(f"Timestamp Parse Error: {e}")
        return None

async def get_db_connection():
    return await asyncpg.connect(**DB_CONFIG)

async def load_posts_from_db(subreddit, limit):
    conn = await get_db_connection()
    try:
        rows = await conn.fetch(
            "SELECT * FROM reddit_posts WHERE subreddit = $1 ORDER BY timestamp DESC LIMIT $2",
            subreddit, limit
        )
        
        data_dict = {}
        for row in rows:
            d = dict(row)
            # 1. Convert timestamp back to string for frontend
            if d.get('timestamp'):
                d['timestamp'] = d['timestamp'].isoformat()
            
            # 2. CRITICAL FIX: Ensure keywords and entities are objects/arrays, not strings
            if isinstance(d.get('keywords'), str):
                d['keywords'] = json.loads(d['keywords'])
            if isinstance(d.get('entities'), str):
                d['entities'] = json.loads(d['entities'])
                
            data_dict[d['id']] = d
        return data_dict
    finally:
        await conn.close()

async def save_post_to_db(post_entry, subreddit):
    conn = await get_db_connection()
    try:
        ts_obj = safe_parse_timestamp(post_entry.get('timestamp'))
        
        await conn.execute('''
            INSERT INTO reddit_posts (id, subreddit, timestamp, title, body, sentiment, keywords, entities)
            VALUES ($1, $2, $3, $4, $5, $6, $7, $8)
            ON CONFLICT (id) DO UPDATE SET
                sentiment = EXCLUDED.sentiment,
                keywords = EXCLUDED.keywords, 
                entities = EXCLUDED.entities
        ''', 
        post_entry['id'], 
        subreddit, 
        ts_obj, 
        post_entry['title'], 
        post_entry['body'], 
        post_entry['sentiment'],
        # Use json.dumps here to satisfy the driver; 
        # Postgres will convert it to JSONB automatically
        json.dumps(post_entry['keywords']), 
        json.dumps(post_entry['entities'])) 
    finally:
        await conn.close()

In [9]:
async def migrate_data():
    cache_dir = "cache"
    if not os.path.exists(cache_dir): return
    
    conn = await get_db_connection()
    print("Migrating JSON files to PostgreSQL...")
    
    for filename in os.listdir(cache_dir):
        if filename.endswith(".json"):
            sub = filename.replace(".json", "")
            with open(os.path.join(cache_dir, filename), "r", encoding="utf-8") as f:
                data = json.load(f)
                for pid, post in data.items():
                    ts_obj = safe_parse_timestamp(post.get('timestamp'))
                    try:
                        await conn.execute('''
                            INSERT INTO reddit_posts (id, subreddit, timestamp, title, body, sentiment, keywords, entities, last_cached)
                            VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9)
                            ON CONFLICT (id) DO NOTHING
                        ''', 
                        pid, sub, ts_obj, post.get('title'), 
                        post.get('body'), post.get('sentiment'), 
                        json.dumps(post.get('keywords', {})), # String for the driver
                        json.dumps(post.get('entities', [])), # String for the driver
                        post.get('last_cached', False))
                    except Exception as e:
                        print(f"Error migrating post {pid}: {e}")
            print(f"Finished r/{sub}")
            
    await conn.close()
    print("Migration Success!")


#await migrate_data()

In [10]:
async def get_post_metadata(post):
    pid = await post.get_attribute("id")
    time_loc = post.locator("time").first
    ts = ""
    if await time_loc.count() > 0:
        ts = await time_loc.get_attribute("datetime")
    return pid, ts

async def scrape_single_post(context, url, pid, ts, websocket, total_priority, tracker, cache, subreddit, is_priority):
    async with semaphore:
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="domcontentloaded", timeout=45000)
            
            title = "[Title Not Found]"
            for sel in ['h1[slot="title"]', 'h1[id^="post-title-"]', 'shreddit-title', 'h1']:
                loc = page.locator(sel).first
                if await loc.is_visible(timeout=3000):
                    title = await loc.inner_text(); break

            content = ""
            for sel in ['shreddit-post-text-body', 'div[slot="text-body"]']:
                loc = page.locator(sel).first
                if await loc.count() > 0:
                    content = await loc.inner_text(timeout=3000)
                    if content: break
            
            # NLP & Sentiment
            full_text = title + " " + content
            sentiment = get_sentiment(full_text)
            words = word_tokenize(full_text.lower())
            sw = set(stopwords.words('english'))
            clean = [lemmatizer.lemmatize(w) for w in words if w.isalnum() and w not in sw and w not in ignore_words]
            keywords = dict(Counter(clean))
            
            # --- PERSIST TO CACHE IMMEDIATELY ---
            post_entry = {
                "id": pid, 
                "timestamp": ts, 
                "title": title, 
                "body": content, 
                "keywords": keywords, 
                "sentiment": sentiment,
                "entities": extract_entities(full_text)
            }
            cache[pid] = post_entry
            await save_post_to_db(post_entry, subreddit)

            # --- SEND DELTAS ONLY FOR PRIORITY NEW POSTS ---
            if is_priority:
                tracker['ui_processed'] += 1
                progress = int((tracker['ui_processed'] / total_priority) * 100) if total_priority > 0 else 100
                
                if websocket.state.name == 'OPEN':
                    # Send the full post object directly
                    await websocket.send(json.dumps({
                        "type": "delta_update",
                        "post": post_entry,
                        "progress": progress
                    }))
                    print(f"New Post Scraped: {title[:30]} | Progress: {progress}%")
            else:
                print(f"Background Cached (Gap Fill): {title[:30]}")
            
            return post_entry
        except Exception as e:
            print(f"Error {url}: {e}"); return None
        finally: await page.close()

In [11]:
async def run_scraper(websocket, subreddit, count, use_only_cache=False):
    count = int(count)
    cache = await load_posts_from_db(subreddit, count)
    
    # --- SHORT CIRCUIT: CACHE ONLY MODE ---
    if use_only_cache:
        print(f"Cache-only mode requested. Bypassing Playwright for r/{subreddit}.")
        if websocket.state.name == 'OPEN':
            await websocket.send(json.dumps({
                "type": "status", 
                "message": f"Loading cached data for r/{subreddit}..."
            }))
            
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            
            await websocket.send(json.dumps({"type": "progress", "value": 100}))
            await websocket.send(json.dumps({
                "type": "final_data", 
                "posts": final_selection
            }))
            await websocket.send(json.dumps({
                "type": "status", 
                "message": "Loaded data exclusively from local cache."
            }))
        return 

    # --- NORMAL MODE: LIVE SCRAPING & DEEP SCROLLING ---
    if websocket.state.name == 'OPEN':
        await websocket.send(json.dumps({
            "type": "status", 
            "message": f"Fetching latest {count} posts from r/{subreddit}..."
        }))

    # Determine checkpoints
    sorted_cache_list = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)
    safe_last_id = next((pid for pid, d in cache.items() if d.get("last_cached")), None)
    if not safe_last_id and sorted_cache_list:
        safe_last_id = sorted_cache_list[0]['id']
    
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False, executable_path=BRAVE_PATH)
    ctx_args = {"storage_state": AUTH_FILE} if os.path.exists(AUTH_FILE) else {}
    context = await browser.new_context(**ctx_args)
    
    try:
        await websocket.send(json.dumps({
            "type": "status", 
            "message": f"Opening r/{subreddit} in the browser..."
        }))
        main_page = await context.new_page()
        await main_page.goto(f"https://www.reddit.com/r/{subreddit}/new", wait_until="domcontentloaded")
        
        launched_ids = set()
        tracker = {'ui_processed': 0}
        top_post_id = None
        
        # --- PHASE 1: CHECK FOR NEW POSTS (Top of Feed) ---
        print("Scanning for new posts...")
        priority_tasks = []
        found_checkpoint = False
        
        while not found_checkpoint:
            posts = await main_page.locator('shreddit-post').all()
            new_on_page = False
            
            for p in posts:
                pid, ts = await get_post_metadata(p)
                if not pid or pid in launched_ids: continue
                if not top_post_id: top_post_id = pid
                
                launched_ids.add(pid)
                new_on_page = True

                if pid == safe_last_id:
                    found_checkpoint = True
                    break

                if pid not in cache:
                    href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                    url = f"https://www.reddit.com{href}"
                    task = asyncio.create_task(
                        scrape_single_post(context, url, pid, ts, websocket, count, tracker, cache, subreddit, True)
                    )
                    priority_tasks.append(task)
            
            if found_checkpoint or not new_on_page: 
                break
            await main_page.mouse.wheel(0, 3000)
            await asyncio.sleep(1)

        if priority_tasks: 
            await asyncio.gather(*priority_tasks)

        # --- PHASE 2: DEEP HISTORY SCROLL (If quota not met) ---
        if len(cache) < count:
            print(f"Quota not met (Cache: {len(cache)} / Target: {count}). Triggering UI update and deep scrolling...")
            
            if websocket.state.name == 'OPEN':
                # 1. Send the data we already have using 'partial_data' so the UI progress bar doesn't break
                current_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)
                await websocket.send(json.dumps({
                    "type": "partial_data", 
                    "posts": current_selection
                }))
                
                # 2. Send the exact requested status message
                await websocket.send(json.dumps({
                    "type": "status", 
                    "message": "Not enough posts in the cache scraping old posts"
                }))

            # 3. Fast-forward the tracker so the delta progress bar math is accurate
            tracker['ui_processed'] = len(cache)
            
            historical_tasks = []
            scroll_attempts = 0
            
            while len(cache) + len(historical_tasks) < count and scroll_attempts < 50:
                posts = await main_page.locator('shreddit-post').all()
                
                for p in posts:
                    pid, ts = await get_post_metadata(p)
                    if not pid or pid in launched_ids: continue
                    
                    launched_ids.add(pid)
                    
                    if pid not in cache:
                        href = await p.locator('a[slot="full-post-link"]').get_attribute('href')
                        url = f"https://www.reddit.com{href}"
                        
                        task = asyncio.create_task(
                            scrape_single_post(context, url, pid, ts, websocket, count, tracker, cache, subreddit, True)
                        )
                        historical_tasks.append(task)
                        
                        if (len(cache) + len(historical_tasks)) >= count: 
                            break
                
                # Force browser to keep scrolling down to load older posts
                await main_page.mouse.wheel(0, 5000)
                await asyncio.sleep(2)
                scroll_attempts += 1
                
                if historical_tasks: 
                    await asyncio.gather(*historical_tasks)
                    historical_tasks = [] 

        # --- PHASE 3: FINAL DELIVERY ---
        if websocket.state.name == 'OPEN':
            final_selection = sorted(cache.values(), key=lambda x: x.get("timestamp") or "", reverse=True)[:count]
            await websocket.send(json.dumps({"type": "progress", "value": 100}))
            await websocket.send(json.dumps({
                "type": "final_data", 
                "posts": final_selection
            }))
            await websocket.send(json.dumps({"type": "status", "message": "Dashboard updated."}))

        # --- PHASE 4: UPDATE CHECKPOINT MARKER ---
        if top_post_id:
            print(f"Caching successful. Moving 'last_cached' marker to {top_post_id} in Database.")
            conn = await get_db_connection()
            try:
                # Reset all markers for this sub
                await conn.execute("UPDATE reddit_posts SET last_cached = FALSE WHERE subreddit = $1", subreddit)
                # Set the new marker
                await conn.execute("UPDATE reddit_posts SET last_cached = TRUE WHERE id = $1", top_post_id)
            finally:
                await conn.close()

    except Exception as e:
        print(f"Playwright Execution Error: {e}")
        print("Process interrupted. 'last_cached' marker remains unchanged to preserve gap state.")
        if websocket.state.name == 'OPEN':
            await websocket.send(json.dumps({
                "type": "status", 
                "message": f"Process got interrupted! - {e}"
            }))
    finally:
        await browser.close()
        await playwright.stop()
        print("Session completed.")

In [12]:
#await reanalyze_cache("LegalAdviceIndia")

In [13]:
async def get_cache_summary():
    """
    Fetches subreddit statistics and formats them as an array of objects:
    [{ subredditname: { count: X, last_updated: Y } }]
    """
    conn = await get_db_connection()
    try:
        # Optimized SQL aggregation
        rows = await conn.fetch('''
            SELECT 
                subreddit, 
                COUNT(*) as post_count, 
                MAX(timestamp) as last_post_time 
            FROM reddit_posts 
            GROUP BY subreddit
        ''')
        
        # Constructing the specific object format
        summary_dict = {}
        for row in rows:
            summary_dict[row['subreddit']] = {
                "count": row['post_count'],
                "last_updated": row['last_post_time'].isoformat() if row['last_post_time'] else None
            }
        return summary_dict
    finally:
        await conn.close()

In [ ]:
async def handler(websocket):
    # --- NEW: Immediate Summary on Connection ---
    print("New connection established. Sending cache summary...")
    try:
        summary_data = await get_cache_summary()
        await websocket.send(json.dumps({
            "type": "cache_summary",
            "message": summary_data
        }))
    except Exception as e:
        print(f"Error sending summary: {e}")

    print("Waiting for command...")
    async for message in websocket:
        data = json.loads(message)
        if data.get('type') == 'start_scrape':
            sub = data.get('subreddit', 'mumbai')
            limit = data.get('count', 10)
            use_only_cache = data.get('useOnlyCache', False)
            
            print(f"Command received: r/{sub} (Limit: {limit}, Cache Only: {use_only_cache})")
            word_counts.clear()
            await run_scraper(websocket, sub, limit, use_only_cache)

async def start_server():
    print("WebSocket Server started on ws://192.168.0.246:8765")
    async with websockets.serve(handler, "192.168.0.246", 8765):
        await asyncio.Future()

await start_server()

WebSocket Server started on ws://192.168.0.246:8765
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Waiting for command...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...
Command received: r/AskIndianWomen (Limit: 500, Cache Only: True)
New connection established. Sending cache summary...
Waiting for command...
Cache-only mode requested. Bypassing Playwright for r/AskIndianWomen.
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Waiting for command...
Error sending summary: receiv

opening handshake failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\http11.py", line 138, in parse
    request_line = yield from parse_line(read_line)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\http11.py", line 320, in parse_line
    line = yield from read_line(MAX_LINE_LENGTH)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\streams.py", line 46, in read_line
    raise EOFError(f"stream ends after {p} bytes, before end of line")
EOFError: stream ends after 0 bytes, before end of line

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\server.py", line 547, in parse
    request = yield from Req

New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
New connection established. Sending cache summary...
Waiting for command...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connec

connection handler failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 376, in conn_handler
    await self.handler(connection)
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_17992\2172116137.py", line 14, in handler
    async for message in websocket:
    ...<8 lines>...
            await run_scraper(websocket, sub, limit, use_only_cache)
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 244, in __aiter__
    yield await self.recv()
          ^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 324, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (going away) WebSocket is closed before the connection is established.
Waiting for command...
Error sending summary:

...

New connection established. Sending cache summary...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Waiting for command...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...


opening handshake failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 356, in conn_handler
    await connection.handshake(
    ...<3 lines>...
    )
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 142, in handshake
    async with self.send_context(expected_state=CONNECTING):
               ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python314\Lib\contextlib.py", line 214, in __aenter__
    return await anext(self.gen)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 965, in send_context
    raise self.protocol.close_exc from original_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


New connection established. Sending cache summary...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Waiting for command...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away); then sent 1001 (going away)
Waiting for command...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: received 1001 (going away) WebSocket is closed before the connection is established.; then sent 1001 (

connection handler failed
Traceback (most recent call last):
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\server.py", line 376, in conn_handler
    await self.handler(connection)
  File "C:\Users\Administrator\AppData\Local\Temp\ipykernel_17992\2172116137.py", line 14, in handler
    async for message in websocket:
    ...<8 lines>...
            await run_scraper(websocket, sub, limit, use_only_cache)
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 244, in __aiter__
    yield await self.recv()
          ^^^^^^^^^^^^^^^^^
  File "d:\React JS\Reddit-Data-Scrapper\Reddit-Scrapper\Lib\site-packages\websockets\asyncio\connection.py", line 324, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedError: no close frame received or sent


New connection established. Sending cache summary...
New connection established. Sending cache summary...
Error sending summary: no close frame received or sent
Waiting for command...
Waiting for command...
New connection established. Sending cache summary...
New connection established. Sending cache summary...
Waiting for command...
Waiting for command...
